In [0]:
from pyspark.sql.functions import current_date, col, udf
from pyspark.sql.types import StringType
import re

# -----------------------------
# Common paths
# -----------------------------
bronze_base_path = "dbfs:/source_to_bronze"
silver_base_path = "dbfs:/silver"
gold_base_path = "dbfs:/gold/employee"

db_name = "employee_info"
silver_table_name = "dim_employee"
gold_table_name = "fact_employee"

# -----------------------------
# Function to write CSV
# -----------------------------
def write_csv_to_dbfs(df, path):
    (
        df.write
          .mode("overwrite")
          .option("header", "true")
          .csv(path)
    )

# -----------------------------
# Function to write Delta table
# -----------------------------
def write_delta_table(df, path, table_name):
    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("path", path)
          .saveAsTable(table_name)
    )

# -----------------------------
# CamelCase to snake_case
# -----------------------------
def camel_to_snake(text):
    s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', text)
    s2 = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)
    return s2.lower()

camel_to_snake_udf = udf(camel_to_snake, StringType())

# -----------------------------
# Rename all columns to snake_case
# -----------------------------
def rename_columns_to_snake(df):
    for old_col in df.columns:
        new_col = camel_to_snake(old_col)
        df = df.withColumnRenamed(old_col, new_col)
    return df